# Exploratory Data Analysis — Profit Drivers

This notebook explores the order-level profitability dataset before any modeling is attempted. The goal is purely descriptive: understand the shape of `net_operating_profit`, check data quality, identify outliers, and see which candidate features look worth testing formally in `02_hypothesis_validation.ipynb`.

No model is fit here — feature selection, model comparison, and driver ranking live in notebook 02.

In [ ]:
import os
import sys
from pathlib import Path

# Resolve project root relative to this notebook's location, rather than
# hardcoding a machine-specific path
def _find_project_root(start: Path, marker: str = "requirements.txt") -> Path:
    for candidate in [start] + list(start.parents):
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate project root (looking for {marker}) above {start}"
    )

PROJECT_ROOT = _find_project_root(Path.cwd())
sys.path.append(str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## 1. Load Data

In [ ]:
from importlib import reload
import src.data_loaders as dl

reload(dl)

df = dl.load_order_profit_raw()
df.shape

## 2. Feature Engineering

Two derived fields are needed before modeling can be tested:
- `shipping_profit`: what's actually recovered on shipping after cost (charged fee minus actual carrier cost)
- `gross_margin_pct`: gross profit as a share of gross revenue, since raw profit figures aren't comparable across order sizes

The candidate feature set below combines these derived fields with cost drivers (discount, marketing cost, return rate) and categorical segments (customer segment, acquisition channel, region, discount band).

In [ ]:
df["shipping_profit"] = df["shipping_fee_charged"] - df["actual_shipping_cost"]
df["gross_margin_pct"] = df["gross_profit"] / df["gross_revenue"]

In [ ]:
TARGET = "net_operating_profit"

features = [
    "discount_pct",
    "shipping_profit",
    "marketing_cost_per_order",
    "unit_return_rate",
    "gross_margin_pct",
    "segment",
    "acquisition_channel",
    "state_region",
    "discount_band"
]

df_model = df[features + [TARGET]].dropna()
df_model.shape

## 3. Descriptive Statistics

In [ ]:
df_model.describe()

In [ ]:
null_counts = df.isnull().sum()
null_counts[null_counts > 0]

### Takeaways

- Most orders generated positive operating profit, though loss-making transactions are present.
- Discounts typically remain low, with most orders receiving 10% or less.
- Shipping profit is negative for the majority of orders, while gross margins remain positive.
- Return rates are generally low, with only a few fully returned orders.

## 4. Distributions

First the target variable, then the numeric candidate features, to check for skew, multi-modality, or scale issues that would affect modeling later.

In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(df_model[TARGET], bins=50)
plt.title("Net Operating Profit Distribution")
plt.show()

In [ ]:
numeric_features = [
    "discount_pct",
    "shipping_profit",
    "marketing_cost_per_order",
    "unit_return_rate",
    "gross_margin_pct"
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, col in zip(axes, numeric_features):
    sns.histplot(df_model[col], bins=40, ax=ax)
    ax.set_title(col)

# Hide any unused subplot
for ax in axes[len(numeric_features):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

## 5. Outlier Detection

Not checked previously. Boxplots flag visually where each numeric feature has values sitting far outside the interquartile range; the IQR table quantifies it as a percentage of rows so it's clear whether an outlier is a handful of orders or a meaningful share of the dataset.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, col in zip(axes, numeric_features):
    sns.boxplot(y=df_model[col], ax=ax)
    ax.set_title(col)

for ax in axes[len(numeric_features):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
def iqr_outlier_pct(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = (series < lower) | (series > upper)
    return outliers.mean() * 100

outlier_summary = pd.DataFrame({
    "feature": numeric_features,
    "pct_outliers_iqr": [iqr_outlier_pct(df_model[c]) for c in numeric_features]
}).sort_values("pct_outliers_iqr", ascending=False)

outlier_summary

### Takeaways

- Marketing cost per order has the highest outlier share (~16.7%), followed by unit return rate (~13.0%).
- Shipping profit contains a smaller number of outliers, while discount percentage shows none under the IQR method.
- These outliers should be considered during modeling but do not appear extensive enough to warrant removal.

## 6. Correlations

In [ ]:
corr_cols = numeric_features + [TARGET]
corr = df_model[corr_cols].corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()

### Takeaways

- Most feature pairs exhibit weak to moderate linear relationships.
- No severe multicollinearity is evident among the predictors.
- Profit appears to be influenced by multiple variables rather than a single dominant feature.

## 7. Bivariate Relationships

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(
    data=df_model.sample(5000),
    x="discount_pct",
    y=TARGET,
    alpha=0.3
)
plt.title("Discount vs Profit")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(
    data=df_model.sample(5000),
    x="unit_return_rate",
    y=TARGET,
    alpha=0.3
)
plt.title("Return Rate vs Profit")
plt.show()

### Takeaways

- Higher discounts are generally associated with lower operating profit.
- Increasing return rates also correspond to lower profitability.
- Both relationships show noticeable variability, indicating additional factors affect profit.

### Summary Insights

- Discounts and return rates show the clearest negative relationship with net operating profit, while cost-related variables also influence profitability.
- Marketing cost per order and unit return rate contain the highest proportion of outliers, though the overall dataset remains suitable for analysis.
- Correlation analysis suggests profit is driven by multiple interacting factors rather than a single dominant variable.
- These exploratory findings provide the basis for validating profit drivers using predictive models in the next notebook.